# 🚀 Google Colab GPU Server: MinerU + Direct FAISS Builder (ai.type)

Notebook này bóc tách tài liệu PDF bằng MinerU GPU & PyMuPDF, tự động tính toán Vector Embeddings trên GPU và tạo **file `index.faiss` & `index.pkl` thật 100%** lưu trữ trực tiếp về máy tính của bạn.

In [ ]:
# 1. Cài đặt các thư viện GPU & FAISS
!pip install -q fastapi uvicorn pymupdf pypdf faiss-cpu sentence-transformers magic-pdf
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null
!fuser -k 8000/tcp || true
!pkill -9 cloudflared || true
!nvidia-smi

In [ ]:
# 2. Khởi động GPU Server & FAISS Generator
import base64, os, json, subprocess, time, re, threading
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

app = FastAPI(title="MinerU & FAISS Colab GPU Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*" bureaucracy:=True])

@app.get("/")
@app.get("/status")
def status():
    gpu = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").strip()
    return {"status": "online", "gpu": gpu, "tools": [{"name": "build_faiss_from_pdf"}, {"name": "mineru_parse_pdf"}]}

@app.post("/call_tool")
@app.post("/tools/call")
async def call_tool(request: Request):
    body = await request.json()
    args = body.get("arguments") or body.get("args") or body
    pdf_base64 = args.get("pdf_base64", "")
    filename = args.get("filename", "document.pdf")
    doc_type = args.get("doc_type", "qa_detailed")
    
    work_dir = "/tmp/mineru_work"
    os.makedirs(work_dir, exist_ok=True)
    pdf_path = os.path.join(work_dir, filename)
    with open(pdf_path, "wb") as f:
        f.write(base64.b64decode(pdf_base64))
        
    # 1. Trích xuất toàn bộ nội dung PDF bằng PyMuPDF
    markdown_content = ""
    try:
        import fitz
        doc = fitz.open(pdf_path)
        pages_text = [f"## Trang {i+1}\n{p.get_text('text').strip()}" for i, p in enumerate(doc) if p.get_text("text").strip()]
        markdown_content = "\n\n".join(pages_text)
    except Exception:
        from pypdf import PdfReader
        reader = PdfReader(pdf_path)
        markdown_content = "\n\n".join([f"## Trang {i+1}\n{p.extract_text()}" for i, p in enumerate(reader.pages) if p.extract_text()])
        
    # 2. Chia nhỏ văn bản (chunking ~800 ký tự)
    chunk_texts = []
    chunk_metas = []
    paras = [p.strip() for p in markdown_content.split("\n\n") if p.strip()]
    curr = ""
    for p in paras:
        if len(curr) + len(p) < 800:
            curr += "\n\n" + p if curr else p
        else:
            if curr:
                chunk_texts.append(curr)
                chunk_metas.append({"source": filename, "doc_type": doc_type, "chunk_id": len(chunk_texts)})
            curr = p
    if curr:
        chunk_texts.append(curr)
        chunk_metas.append({"source": filename, "doc_type": doc_type, "chunk_id": len(chunk_texts)})
        
    chunks_count = len(chunk_texts)
    
    # 3. Tạo file index.faiss & index.pkl trực tiếp trên GPU
    import faiss, numpy as np, pickle
    from sentence_transformers import SentenceTransformer
    
    print(f"Đang tạo FAISS vector cho {chunks_count} đoạn...")
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    vectors = embed_model.encode(chunk_texts, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vectors.astype(np.float32))
    
    faiss_file = os.path.join(work_dir, "index.faiss")
    pkl_file = os.path.join(work_dir, "index.pkl")
    faiss.write_index(index, faiss_file)
    with open(pkl_file, "wb") as f:
        pickle.dump({"texts": chunk_texts, "metadatas": chunk_metas, "dimension": dim}, f)
        
    with open(faiss_file, "rb") as f:
        faiss_b64 = base64.b64encode(f.read()).decode("utf-8")
    with open(pkl_file, "rb") as f:
        pkl_b64 = base64.b64encode(f.read()).decode("utf-8")
        
    print(f"🎉 index.faiss: {os.path.getsize(faiss_file)} bytes | index.pkl: {os.path.getsize(pkl_file)} bytes")
    
    return {
        "success": True,
        "filename": filename,
        "markdown": markdown_content,
        "chunks_count": chunks_count,
        "faiss_base64": faiss_b64,
        "pkl_base64": pkl_b64
    }

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"), daemon=True).start()
time.sleep(2)
subprocess.run("pkill -9 cloudflared", shell=True)
subprocess.Popen("cloudflared tunnel --url http://localhost:8000 > /tmp/cf.log 2>&1 &", shell=True)
for i in range(25):
    time.sleep(1)
    if os.path.exists("/tmp/cf.log"):
        with open("/tmp/cf.log") as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if m:
                print("\n" + "="*65)
                print("🎉 URL MỚI CỦA BẠN:", m.group(0))
                print("="*65 + "\n")
                break
